In [1]:
from utils import *
import pickle
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import KernelPCA
from sklearn.preprocessing import StandardScaler
import os
os.environ['OMP_NUM_THREADS'] = '3'

with open("embeddings_normalized__niigz_.pkl", "rb") as f:
    embeddings_df = pickle.load(f)
    
def get_cluster_labels(embeddings_dict, n_clusters):
    # stack series of arrays into 2D array
    embeddings = np.stack(embeddings_dict['cls'].values)  # (N, D)
    print(f"Embeddings shape: {embeddings.shape}")

    # normalize
    X = (embeddings)

    # # KPCA for reducing dimensions while preserving structure
    # n_components = min(50, X.shape[0] - 1, X.shape[1])
    # kpca = KernelPCA(n_components=n_components, kernel='cosine', random_state=42)
    # X_kpca = kpca.fit_transform(X)

    # # KMeans clustering
    # n_clusters = max(2, n_clusters)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(X)

    keys = list(embeddings_dict.index)  # use dataframe index as keys
    return {key: label for key, label in zip(keys, kmeans.labels_)}

LABELLED_DATA_PERCENTAGE = 0.1

psma_df = embeddings_df[embeddings_df['filename'].str.contains('psma', case=False)]
fdg_df  = embeddings_df[embeddings_df['filename'].str.contains('fdg',  case=False)]

print(f"PSMA samples: {len(psma_df)}")
print(f"FDG  samples: {len(fdg_df)}")



PSMA samples: 379
FDG  samples: 664


In [56]:
len(psma_df)

379

In [2]:
import numpy as np
from typing import List, Optional

def dpp_selection(
    embeddings: np.ndarray,
    num_select: int,
    quality_scores: Optional[np.ndarray] = None
) -> List[int]:
    """
    Select diverse subset using Determinantal Point Process.
    
    Args:
        embeddings: Shape (N, 768) - your embedding vectors
        num_select: How many items you want back
        quality_scores: Optional (N,) array of quality/relevance scores
                       If None, all items treated as equal quality
    
    Returns:
        List of selected indices
    """
    
    # ── 1. Normalize embeddings ──────────────────────────────────────
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalized = embeddings / (norms + 1e-10)
    
    # ── 2. Build Similarity Kernel (cosine similarity matrix) ────────
    S = normalized @ normalized.T  # Shape: (N, N)
    
    # ── 3. Incorporate quality scores if provided ────────────────────
    if quality_scores is not None:
        # Scale quality scores to [0, 1]
        q = quality_scores / (quality_scores.max() + 1e-10)
        # L_ij = q_i * S_ij * q_j  (quality-weighted kernel)
        L = np.outer(q, q) * S
    else:
        L = S
    
    # ── 4. Greedy MAP Inference ──────────────────────────────────────
    #    (exact DPP sampling is expensive for large N, greedy is practical)
    selected = []
    remaining = list(range(len(embeddings)))
    
    for _ in range(num_select):
        if not remaining:
            break
            
        best_idx = None
        best_score = -np.inf
        
        if len(selected) == 0:
            # First item: just pick highest quality / diagonal element
            scores = np.array([L[i, i] for i in remaining])
            best_idx = remaining[np.argmax(scores)]
            
        else:
            # Pick item that maximizes determinant gain
            # det gain ∝ L[i,i] - L[i,S] @ inv(L[S,S]) @ L[S,i]
            S_idx = np.array(selected)
            
            L_SS = L[np.ix_(S_idx, S_idx)]
            L_SS_inv = np.linalg.inv(L_SS + 1e-6 * np.eye(len(S_idx)))
            
            for i in remaining:
                L_iS = L[i, S_idx]
                # Schur complement = marginal gain in det
                schur = L[i, i] - L_iS @ L_SS_inv @ L_iS.T
                if schur > best_score:
                    best_score = schur
                    best_idx = i
        
        selected.append(best_idx)
        remaining.remove(best_idx)
    
    return selected

In [9]:
psma_embeddings = np.stack(psma_df['cls'].values)  # (N, D)
fdg_embeddings  = np.stack(fdg_df['cls'].values)   # (N
psma_embeddings.shape, fdg_embeddings.shape

((379, 768), (664, 768))

In [15]:
psma_df

,filename,cls,patch_mean,patch_std
0,psma_7e76841f3c9623c0_2016-01-23_0001.nii.gz,"[2.5620344, 2.6604629, 1.894331, 1.2153169, -0...","[0.050740976, -0.3065505, -0.30072844, 0.61455...","[1.6356311, 1.342394, 1.4462588, 1.2509305, 1...."
7,psma_69ea0c011af2e2d4_2016-07-29_0001.nii.gz,"[2.2795327, 3.8346975, 2.652379, 2.3553536, -1...","[-0.16908371, -0.21220593, -0.047096558, 0.475...","[1.6173371, 1.4222492, 1.1875254, 1.1890163, 1..."
15,psma_823a3a884418928b_2019-12-23_0001.nii.gz,"[2.3693936, 3.8688874, 3.1879885, 2.0962412, -...","[0.34644732, -0.20004575, -1.0308343, -0.11137...","[1.7865598, 1.3557264, 1.2208041, 1.3914933, 1..."
20,psma_56e540714db4775a_2020-03-27_0001.nii.gz,"[2.3638287, 2.967399, 1.212798, 0.38169152, -1...","[0.49855417, 0.21621045, -0.23461133, -0.08085...","[1.5731763, 1.4911697, 1.2123907, 1.0369339, 1..."
21,psma_3b06da5b6cd14d9d_2015-11-27_0001.nii.gz,"[0.7575063, 1.1707925, 2.257357, 2.038772, -0....","[-0.36760995, -0.73012984, -0.61694735, 0.5524...","[1.5064524, 1.2616304, 1.0826173, 1.225331, 2...."
...,...,...,...,...
1034,psma_605d8fc340cd0e00_2017-07-31_0001.nii.gz,"[0.15913455, 2.7602808, 2.671065, 2.3957474, -...","[-0.15885459, -0.01174563, -0.36595145, 0.8212...","[1.8377843, 1.334458, 1.2697015, 1.4114871, 1...."
1035,psma_0cd58730e7446445_2016-06-11_0001.nii.gz,"[0.8593852, 4.3346624, 2.8853903, 2.2676501, -...","[0.16770516, 0.17270781, 0.14799348, 0.1719474...","[1.9928356, 1.3868188, 1.2443005, 1.3080024, 1..."
1037,psma_ca36bd95b63289d0_2020-05-15_0001.nii.gz,"[2.708599, 2.0376875, 2.719223, 1.1814945, -2....","[0.5958748, 0.1772778, -0.27325517, -0.7678045...","[1.868466, 1.1998976, 1.3851563, 1.3962709, 2...."
1039,psma_fb794d977655f3c5_2020-10-05_0001.nii.gz,"[1.4691865, 4.382557, 2.5568085, 2.9033315, -1...","[0.4215832, 0.100460485, 0.2791628, 0.64271665...","[1.7523737, 1.5190547, 1.219234, 1.2857175, 1...."


In [22]:
# get the name of the psma_indices
psma_cases_names = psma_df.iloc[psma_indices]['filename'].values
fdg_case_names  = fdg_df.iloc[fdg_indices]['filename'].values

# just filter the names correctly 
psma_case_names = [name.replace('_0001.nii.gz', '') for name in psma_cases_names]
fdg_case_names = [name.replace('_0001.nii.gz', '') for name in fdg_case_names]

In [66]:

import json


num_psma_cases = [37, 55, 75, 113, 151, 265]
num_fdg_cases  = [66, 99, 132, 199, 265, 464]

def get_dpp_psma_fdg(num_psma, num_fdg): 
    psma_indices = dpp_selection(psma_embeddings, num_select=num_psma)
    fdg_indices  = dpp_selection(fdg_embeddings,  num_select=num_fdg)
    
    psma_cases_names = psma_df.iloc[psma_indices]['filename'].values
    fdg_case_names  = fdg_df.iloc[fdg_indices]['filename'].values
    
    psma_case_names = [name.replace('_0001.nii.gz', '') for name in psma_cases_names]
    fdg_case_names = [name.replace('_0001.nii.gz', '') for name in fdg_case_names]
    
    return psma_case_names, fdg_case_names

with open('new_AL_splits.json', 'r') as f:
    old_splits = json.load(f)
val_files = old_splits[0]['val']


dpp_based_splits = []
dpp_based_splits.append(old_splits[0]) # original 10% split 

for num_psma, num_fdg in zip(num_psma_cases, num_fdg_cases):
    psma_case_names, fdg_case_names = get_dpp_psma_fdg(num_psma, num_fdg)
    train_files = psma_case_names + fdg_case_names
    val_files = old_splits[0]['val']  # original val set (no overlap with train)
    dpp_based_splits.append({
        'train': train_files,
        'val': val_files
    })

with open('dpp_based_splits.json', 'w') as f:
    json.dump(dpp_based_splits, f, indent=4)
    




In [68]:
dpp_added_splits = []

embeds_filtered = embeddings_df[~embeddings_df['filename'].str.replace('_0001.nii.gz', '').isin(old_splits[0]['train'])]
psma_filtered = embeds_filtered[embeds_filtered['filename'].str.contains('psma', case=False)]
fdg_filtered  = embeds_filtered[embeds_filtered['filename'].str.contains('fdg',  case=False)]

for num_psma, num_fdg in list(zip(num_psma_cases, num_fdg_cases))[1:]:
    # subtract the cases in the intial 10% split
    psma_case_names, fdg_case_names = get_dpp_psma_fdg(num_psma - 37, num_fdg - 66)
    train_files = psma_case_names + fdg_case_names + old_splits[0]['train'] # original cases
    val_files = old_splits[0]['val']  # original val set (no overlap with train)
    dpp_added_splits.append({
        'train': train_files,
        'val': val_files
    })

with open('dpp_added_splits.json', 'w') as f:
    json.dump(dpp_added_splits, f, indent=4)


103
Train size: 154, Val size: 247
103
Train size: 207, Val size: 247
103
Train size: 312, Val size: 247
103
Train size: 416, Val size: 247
103
Train size: 729, Val size: 247


In [ ]:
import json 
# import old splits file for validation cases

with open('new_AL_splits.json', 'r') as f:
    old_splits = json.load(f)
val_files = old_splits[0]['val']

for split in old_splits:
    try: 
        #print(len(split['train']), len(split['val']))
        pass
    except Exception as e:
        pass
 
original_split_5 = old_splits[5]
# check intersection
inter = (set(psma_case_names) | set(fdg_case_names)) & set(original_split_5['train'])    

len(inter)

315

In [43]:
len(old_splits[0]['train']), len(old_splits[0]['val'])

(103, 247)